In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr
from sklearn.metrics import r2_score

In [ ]:
# [THEME SETUP]
# Use a clean, academic style (white background, gridlines for readability)
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams.update({
    'figure.titlesize': 16,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.family': 'sans-serif'
})

In [ ]:
def format_plot(ax, title, xlabel, ylabel):
    """
    Applies the Academic Standard formatting to any Axes object.
    Ensures every plot in your paper looks identical.
    """
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel(xlabel, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=12, fontweight='bold')

    # Remove the top and right spines (The "Academic Clean Look")
    sns.despine(ax=ax)

    # Ensure grid is below the data
    ax.set_axisbelow(True)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

def format_plot_horizontal(ax, title, xlabel, ylabel):
    """
    Specialized formatting for Horizontal Plots (Boxplots/Barh).
    - Grid: Vertical (axis='x')
    - Spines: Removes Top and Right
    """
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel(xlabel, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=12, fontweight='bold')

    # Standard "Clean Look"
    sns.despine(ax=ax)

    # CRITICAL CHANGE: Grid must be on X-axis for horizontal plots
    ax.set_axisbelow(True)
    ax.grid(axis='x', linestyle='--', alpha=0.5)

In [ ]:
def cal_Obesity(BMI):
    return abs((BMI/22-1)*100)

In [ ]:
filePath = '../Dataset/UCI_Gallstone_Dataset.csv'
df = pd.read_csv(filePath)
# df['WaterVar']=df['Total Body Water (TBW)']-df['Extracellular Water (ECW)']-df['Intracellular Water (ICW)']
# df['area/mass']= df['Visceral Muscle Area (VMA) (Kg)']/df['Muscle Mass (MM)']
df

In [ ]:
df.isnull().values.any()

In [ ]:
df.info()

In [ ]:
df.describe().transpose()

In [ ]:
# [GROUP 1] Demographic & Anthropometric Baselines
# Method: Electronic Health Records (EHR) & Standard Physical Measurement.
# - Demographics: Age (Years) and Gender (Binary: 0=Male, 1=Female).
# - Anthropometry: Height (cm), Weight (kg), and Body Mass Index (kg/m²).
df_demographic = df[[
    'Age',
    'Gender',
    'Height',
    'Weight',
    'Body Mass Index (BMI)'
]].copy()

# [GROUP 2] Clinical History & Ultrasound Grading
# Method: Medical History Review (Binary) & Ultrasonography (Ordinal).
# - Comorbidity Level: Dataset contains values 0-3 (Score/Class).
#   *Note: The paper describes this as binary, but the data indicates a severity score or count.
# - Specific Conditions: Binary flags (1=Yes, 0=No) for CAD, Hypothyroidism, Hyperlipidemia, and Diabetes [cite: 68-70].
# - Hepatic Fat Accumulation (HFA): Fatty liver severity (Grades 0-3) diagnosed specifically via Ultrasonography.

df_comorbiditiesCount = df[[
    'Comorbidity' # Ordinal Score (0-3) in CSV vs Binary in Paper.
]].copy()

df_comorbidity = df[[
    'Coronary Artery Disease (CAD)', # Binary (0/1)
    'Hypothyroidism',                # Binary (0/1)
    'Hyperlipidemia',                # Binary (0/1)
    'Diabetes Mellitus (DM)',        # Binary (0/1)
]].copy()

df_HepaticFatAccumulation = df[[
    'Hepatic Fat Accumulation (HFA)' # Ordinal Grade (0-3) via Ultrasound.
]].copy()

# [GROUP 3] Bioelectrical Impedance Analysis (BIA) & Imaging
# Method A (Composition): "Tanita MC780" Body Composition Analyzer[cite: 74].
# - Protocol: Non-invasive electrical resistance measurement of tissues.
# - Metrics: TBW (kg), Fat (%), Muscle (kg), Bone (kg).
#
# Method B (Liver Fat): "Ultrasonography"[cite: 77].
# - Hepatic Fat Accumulation (HFA): Diagnosed via imaging, not BIA.
df_bioimpedance = df[[
    'Total Body Water (TBW)',
    'Extracellular Water (ECW)',
    'Intracellular Water (ICW)',
    'Extracellular Fluid/Total Body Water (ECF/TBW)',
    'Total Body Fat Ratio (TBFR) (%)',
    'Lean Mass (LM) (%)',
    'Body Protein Content (Protein) (%)',
    'Visceral Fat Rating (VFR)',
    'Bone Mass (BM)',
    'Muscle Mass (MM)',
    'Obesity (%)',
    'Total Fat Content (TFC)',
    'Visceral Fat Area (VFA)',
    'Visceral Muscle Area (VMA) (Kg)'
]].copy()

# [GROUP 4] Laboratory & Metabolic Markers
# Method: Clinical "Blood Tests" routinely gauged in laboratories.
# Lipid/Glucose: Measured in mg/dL (Glucose, Cholesterol, LDL, HDL, Triglycerides).
# Liver Enzymes: Measured in U/L (AST, ALT, ALP).
# Renal Function: Creatinine (mg/dL) and GFR (ml/seconds).
# Specialized: CRP (mg/L), Hemoglobin (g/dL), and vitamin D (ng/mL).
df_laboratory = df[[
    'Glucose',
    'Total Cholesterol (TC)',
    'Low Density Lipoprotein (LDL)',
    'High Density Lipoprotein (HDL)',
    'Triglyceride',
    'Aspartat Aminotransferaz (AST)',
    'Alanin Aminotransferaz (ALT)',
    'Alkaline Phosphatase (ALP)',
    'Creatinine',
    'Glomerular Filtration Rate (GFR)',
    'C-Reactive Protein (CRP)',
    'Hemoglobin (HGB)',
    'Vitamin D'
]].copy()

In [ ]:
# 1. Define all data slices in a dictionary
data_groups = {
    "1. Demographic": df_demographic,
    "2a. Comorb. (Score)": df_comorbiditiesCount,
    "2b. Comorb. (Binary)": df_comorbidity,
    "2c. HFA (Target/Grade)": df_HepaticFatAccumulation,
    "3. Bioimpedance": df_bioimpedance,
    "4. Laboratory": df_laboratory
}

# 2. Print Dimensions
print(f"{'GROUP SUBSET':<30} | {'SHAPE (Rows, Cols)'}")
print("-" * 55)

total_selected_features = 0

for name, data in data_groups.items():
    print(f"{name:<30} | {data.shape}")
    total_selected_features += data.shape[1]

print("-" * 55)

# 3. Validation Logic
# The goal is to compare Total Selected vs (Original - 1)
original_cols = df.shape[1]
target_cols = original_cols - 1

print(f"Total Selected Columns:       {total_selected_features}")
print(f"Original DataFrame Columns:   {original_cols}")
print(f"Target Count (Original - 1):  {target_cols}")

print("-" * 55)

if total_selected_features == target_cols:
    print("✅ SUCCESS: Selected features match 'Original - 1'.")
elif total_selected_features == original_cols:
    print("⚠️ NOTE: Selected count equals Original (Target likely included in selection).")
else:
    diff = total_selected_features - target_cols
    print(f"❌ MISMATCH: Difference of {diff} columns.")

In [ ]:
df_demographic['Gender_Label'] = df_demographic['Gender'].map({0: 'Male', 1: 'Female'})

# PLOT GENERATION
g = sns.pairplot(
    df_demographic,
    hue='Gender_Label',
    vars=['Age', 'Height', 'Weight', 'Body Mass Index (BMI)'],
    kind='reg',
    diag_kind='kde',
    palette='husl',

    # 1. FIX: Removed 'color': 'black' so lines match the Gender color
    plot_kws={
        'scatter_kws': {'alpha': 0.4, 's': 20},
        'line_kws': {'linewidth': 1.5}
    },

    # 2. FIX: Changed 'shade' to 'fill' to resolve FutureWarning
    diag_kws={'fill': True, 'alpha': 0.3},

    height=2.5,
    aspect=1.2
)

g.fig.suptitle('[GROUP 1] Demographic Baselines: Distribution & Correlation', y=1.02, fontsize=16)

plt.show()

In [ ]:
# --- PLOT: Comorbidity Score Distribution (Refactored) ---
fig, ax = plt.subplots(figsize=(10, 6))

# [Data Prep Logic remains the same...]
counts = df_comorbiditiesCount['Comorbidity'].value_counts()
max_score = int(df_comorbiditiesCount['Comorbidity'].max())
plot_data = counts.reindex(range(max_score + 1), fill_value=0)

# 1. Plot
sns.barplot(
    x=plot_data.index.astype(int),
    y=plot_data.values,
    hue=plot_data.index.astype(int),
    legend=False,
    ax=ax,
    palette='Blues',
    edgecolor='black',
    width=0.6
)

# 2. APPLY THE STANDARD (The Fix)
# Instead of 6 lines of manual formatting, we use 1 line:
format_plot(
    ax=ax,
    title='Distribution of Comorbidity Scores',
    xlabel='Score / Class (Severity)',
    ylabel='Patient Count'
)

# 3. Post-Standard Adjustments (Specific to this plot)
# Since format_plot handles the grid/spines/labels, you only add unique tweaks here:
ax.set_ylim(0, plot_data.max() * 1.2)

# Uniform Labels (Keep your existing logic)
for container in ax.containers:
    ax.bar_label(container, padding=4, color='#D32F2F', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# --- PLOT: Prevalence of Specific Conditions (Vertical Correction) ---
fig, ax = plt.subplots(figsize=(10, 6))

# 1. Data Processing
prevalence = df_comorbidity.sum().sort_values(ascending=True)

# 2. The "Signal" Plot (Vertical)
# CHANGE: Use 'bar' instead of 'barh'
bars = ax.bar(
    prevalence.index,        # X-Axis: The Classes
    prevalence.values,       # Y-Axis: The Counts
    color='#C0392B',
    edgecolor='black',
    width=0.6
)

# 3. Apply the Standard
format_plot(
    ax=ax,
    title='Prevalence of Comorbidities (Positive Cases)',
    xlabel='Condition',         # Label change
    ylabel='Number of Patients' # Label change
)

# 4. Aesthetics (Crucial for Vertical Plots)
# Rotate labels 15 degrees so they don't crash into each other
plt.xticks(rotation=15, ha='center')

# Add values on top of the bars
ax.bar_label(
    bars,
    padding=3,
    fontweight='bold',
    color='#333333'
)

# 5. Headroom
# Switch from set_xlim to set_ylim for vertical headroom
ax.set_ylim(0, prevalence.max() * 1.15)

plt.tight_layout()
plt.show()

In [ ]:
# --- PLOT: Hepatic Fat Accumulation (Raw Distribution) ---
fig, ax = plt.subplots(figsize=(8, 6))

# 1. Prepare Data (Original - No Filtering)
# We strictly plot what exists in the CSV.
plot_data = df_HepaticFatAccumulation['Hepatic Fat Accumulation (HFA)'].value_counts().sort_index()

# 2. Plot
# We need 5 colors now (Grades 0, 1, 2, 3, and the Anomalous 4)
# We use a sequential palette, but Grade 4 will naturally appear as "Maximum Severity"
colors = plt.cm.viridis([0.1, 0.3, 0.5, 0.7, 0.9])

bars = ax.bar(
    plot_data.index.astype(str),
    plot_data.values,
    color=colors,
    edgecolor='black',
    width=0.6
)

# 3. Apply Standard
format_plot(
    ax=ax,
    title='Hepatic Fat Accumulation (HFA) Severity',
    xlabel='Ultrasound Grade (Raw Data)',
    ylabel='Patient Count'
)
ax.bar_label(bars, padding=3, fontweight='bold', color='#333333')

# 4. Contextual Legend
# NOTE: The paper  defines HFA via Ultrasonography (Standard Grades 0-3).
# 'Grade 4' appears in the dataset but is undefined in the protocol.
# We retain it for analytics integrity as per instruction.
legend_text = (
    "Severity Scale:\n"
    "■ Grade 0: Normal\n"
    "■ Grade 1: Mild\n"
    "■ Grade 2: Moderate\n"
    "■ Grade 3: Severe\n"
    "■ Grade 4: Undefined*"
)

ax.text(
    0.95, 0.95, legend_text,
    transform=ax.transAxes,
    fontsize=10,
    verticalalignment='top',
    horizontalalignment='right',
    multialignment='left', # Aligns text to the left side of the box
    bbox=dict(boxstyle="round,pad=0.5", facecolor='white', alpha=0.9, edgecolor='gray')
)

# 5. Annotation for the Anomaly (Optional Visual Aid)
# Only if Grade 4 exists, we add a small red asterisk or note near the X-axis label
if 4 in plot_data.index:
    # Find the x-position of the '4' bar (which is index 4)
    # We add a subtle footnote
    plt.figtext(0.5, 0.01, "* Grade 4 is present in raw data but inconsistent with standard medical grading (0-3).",
                ha="center", fontsize=9, color="#C0392B", style='italic')

plt.tight_layout()
plt.show()

In [ ]:
import math
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Configuration
features_to_plot = df_bioimpedance.columns
cols = 3
n_features = len(features_to_plot)
rows = math.ceil(n_features / cols)

# 2. Setup Canvas
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
fig.suptitle('Bioimpedance Features: Distribution Analysis', fontsize=18, y=1.02)

# Flatten axes for easy iteration
axes_flat = axes.flatten()

# 3. Plotting Loop
for i, col_name in enumerate(features_to_plot):
    ax = axes_flat[i]

    # [Correction 1] Unit Extraction
    # We split "Visceral Muscle Area (VMA) (Kg)" into Title and X-Label
    if '(' in col_name:
        clean_title = col_name.split('(')[0].strip()
        # Takes everything from the first '(' onwards as the unit label
        unit_text = col_name[col_name.find('('):]
    else:
        clean_title = col_name
        unit_text = ""

    # [Correction 2] The Plot (All Histograms as requested)
    sns.histplot(
        data=df_bioimpedance,
        x=col_name,
        kde=True,
        ax=ax,
        color='skyblue',
        edgecolor='black'
    )

    # [Correction 3] Applying Your Standard
    # We pass the arguments strictly as your function requires (ax, title, xlabel, ylabel)
    format_plot(
        ax,
        clean_title,
        unit_text,
        'Frequency'
    )

# 4. Cleanup: Delete empty subplots
for j in range(i + 1, len(axes_flat)):
    fig.delaxes(axes_flat[j])

plt.tight_layout()
plt.show()

In [ ]:
# 1. Configuration
features_to_plot = df_bioimpedance.columns
cols = 3
n_features = len(features_to_plot)
rows = math.ceil(n_features / cols)

# 2. Setup Canvas
fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows))
fig.suptitle('Bioimpedance Features: Outlier Detection (Horizontal)', fontsize=18, y=1.02)

axes_flat = axes.flatten()

# 3. Plotting Loop
for i, col_name in enumerate(features_to_plot):
    ax = axes_flat[i]

    # [Standardization] Unit Extraction
    if '(' in col_name:
        clean_title = col_name.split('(')[0].strip()
        unit_text = col_name[col_name.find('('):]
    else:
        clean_title = col_name
        unit_text = ""

    # Plot: Horizontal Boxplot
    sns.boxplot(
        data=df_bioimpedance,
        x=col_name,  # Metric on X-axis (Numeric)
        ax=ax,
        color='#F1948A', # Salmon color
        flierprops={"marker": "x", "color": "black", "markersize": 5},
        width=0.6
    )

    # Apply Horizontal Standard directly here
    ax.set_title(clean_title, fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel(unit_text, fontsize=10, fontweight='bold')
    ax.set_ylabel('', fontsize=10, fontweight='bold') # Empty Y-label

    sns.despine(ax=ax)
    ax.set_axisbelow(True)
    ax.grid(axis='x', linestyle='--', alpha=0.5) # Grid on X for horizontal plots

# 4. Cleanup
for j in range(i + 1, len(axes_flat)):
    fig.delaxes(axes_flat[j])

plt.tight_layout()
plt.show()

In [ ]:
# 1. Configuration
features_to_plot = df_laboratory.columns
cols = 3
n_features = len(features_to_plot)
rows = math.ceil(n_features / cols)

# 2. Setup Canvas
fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows)) # Standardized Width
fig.suptitle('Laboratory Markers: Distribution Analysis', fontsize=18, fontweight='bold', y=1.02)

axes_flat = axes.flatten()

# 3. Plotting Loop
for i, col_name in enumerate(features_to_plot):
    ax = axes_flat[i]

    # [Standardization] Unit Extraction
    # Example: "Glucose (mg/dL)" -> Title: "Glucose", Unit: "(mg/dL)"
    if '(' in col_name:
        clean_title = col_name.split('(')[0].strip()
        unit_text = col_name[col_name.find('('):]
    else:
        clean_title = col_name
        unit_text = ""

    # Plot: Histogram + KDE
    sns.histplot(
        data=df_laboratory,
        x=col_name,
        kde=True,
        ax=ax,
        color='#76D7C4', # Medical Green
        edgecolor='black',
        line_kws={'linewidth': 1.5}
    )

    # [Standardization] Apply Your Vertical Standard
    format_plot(
        ax,
        clean_title,
        unit_text,    # Unit goes on X-axis for Histograms
        'Frequency'   # Y-axis is Count/Freq
    )

    # [Optional] Add Mean Marker (The "Normal" Anchor)
    # Useful for Lab tests to see where the average patient sits
    mean_val = df_laboratory[col_name].mean()
    ax.axvline(mean_val, color='#C0392B', linestyle='--', linewidth=1.5, alpha=0.8)

# 4. Cleanup
for j in range(i + 1, len(axes_flat)):
    fig.delaxes(axes_flat[j])

plt.tight_layout()
plt.show()

In [ ]:
features_to_plot = df_laboratory.columns
cols = 3
n_features = len(features_to_plot)
rows = math.ceil(n_features / cols)

# Setup Canvas
fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows))
fig.suptitle('Laboratory Markers: Outlier Detection (Boxplots)', fontsize=18, fontweight='bold', y=1.02)

axes_flat = axes.flatten()

# Plotting Loop
for i, col_name in enumerate(features_to_plot):
    ax = axes_flat[i]

    # [Standardization] Unit Extraction
    if '(' in col_name:
        clean_title = col_name.split('(')[0].strip()
        unit_text = col_name[col_name.find('('):]
    else:
        clean_title = col_name
        unit_text = ""

    # Plot: Horizontal Boxplot
    sns.boxplot(
        data=df_laboratory,
        x=col_name,
        ax=ax,
        color='#F1948A', # Academic Salmon (Consistent with Bioimpedance Outliers)
        flierprops={"marker": "x", "color": "#C0392B", "markersize": 5, "markeredgewidth": 1.5}, # Red 'x'
        width=0.6
    )

    # [Standardization] Apply Horizontal Format
    format_plot_horizontal(
        ax,
        clean_title,
        unit_text,   # Unit on X-axis
        ''           # Empty Y-axis
    )

# Cleanup
for j in range(i + 1, len(axes_flat)):
    fig.delaxes(axes_flat[j])

plt.tight_layout()
plt.show()

In [ ]:
# 1. DATA PREP
df_orig = df.copy()
df_orig['Calculated_Obesity'] = df_orig['Body Mass Index (BMI)'].apply(cal_Obesity)
df_clean = df_orig[~np.isclose(df_orig['Obesity (%)'], 1954.00)].copy()

# --- METRIC 1: REGRESSION SCORE (Consistency) ---
# We square the Pearson R to get R^2 for the regression line
r_orig, _ = pearsonr(df_orig['Calculated_Obesity'], df_orig['Obesity (%)'])
r2_regr_orig = r_orig**2

r_clean, _ = pearsonr(df_clean['Calculated_Obesity'], df_clean['Obesity (%)'])
r2_regr_clean = r_clean**2

# --- METRIC 2: THEORETICAL SCORE (Accuracy) ---
# How well does it fit the PHYSICS FORMULA (y=x)?
def calc_theoretical_r2(df_input):
    # Order matters: (True, Pred) -> (Sensor, Formula)
    # This measures how well the formula predicts the sensor
    return r2_score(df_input['Obesity (%)'], df_input['Calculated_Obesity'])

r2_theory_orig = calc_theoretical_r2(df_orig)
r2_theory_clean = calc_theoretical_r2(df_clean)

# 2. VISUALIZATION
fig, axes = plt.subplots(2, 2, figsize=(16, 12), layout='constrained')

LABEL_Y_SENSOR = "Measured Obesity Degree (Sensor)"
LABEL_X_FORMULA = "Calculated Obesity Degree (Formula)"

# ==========================================
# ROW 1: SCENARIO A (Original)
# ==========================================
# 1A. Boxplot
sns.boxplot(data=df_orig[['Obesity (%)']], orient='h', ax=axes[0, 0], color='#C0392B', width=0.4)
format_plot_horizontal(axes[0, 0], 'A1. Sensor Distribution (With Error)', LABEL_Y_SENSOR, '')

# 1B. Regression
sns.regplot(
    data=df_orig, x='Calculated_Obesity', y='Obesity (%)', ax=axes[0, 1],
    color='#C0392B', scatter_kws={'alpha': 0.5, 's': 30},
    line_kws={'color': 'black', 'label': 'Regression Trend'}
)
# Outlier Annotation
outlier_row = df_orig[np.isclose(df_orig['Obesity (%)'], 1954.00)]
if not outlier_row.empty:
    x_out, y_out = outlier_row['Calculated_Obesity'].values[0], outlier_row['Obesity (%)'].values[0]
    axes[0, 1].annotate("Sensor Error", xy=(x_out, y_out), xytext=(x_out-40, y_out-300),
                        arrowprops=dict(facecolor='black', shrink=0.05), fontsize=10,
                        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.9))

# [CHANGE] Unified R² Metrics
title_a = (f"A2. Original Correlation: FAILED\n"
           f"Regression Fit ($R^2$): {r2_regr_orig:.4f}  |  Theoretical Match ($R^2$): {r2_theory_orig:.4f}")
format_plot(axes[0, 1], title_a, LABEL_X_FORMULA, LABEL_Y_SENSOR)


# ==========================================
# ROW 2: SCENARIO B (Cleaned)
# ==========================================
# 2A. Boxplot
sns.boxplot(data=df_clean[['Obesity (%)']], orient='h', ax=axes[1, 0], color='#5DADE2', width=0.4)
format_plot_horizontal(axes[1, 0], 'B1. Sensor Distribution (Cleaned)', LABEL_Y_SENSOR, '')

# 2B. Regression
sns.regplot(
    data=df_clean, x='Calculated_Obesity', y='Obesity (%)', ax=axes[1, 1],
    color='teal', scatter_kws={'alpha': 0.5, 's': 30},
    line_kws={'color': '#E67E22', 'linewidth': 3, 'label': 'Regression Trend'}
)
limit = max(df_clean['Obesity (%)'].max(), df_clean['Calculated_Obesity'].max())
axes[1, 1].plot([0, limit], [0, limit], color='gray', linestyle='--', linewidth=1.5, label='Theoretical Match (y=x)')

# [CHANGE] Unified R² Metrics
title_b = (f"B2. Cleaned Correlation: CONFIRMED\n"
           f"Regression Fit ($R^2$): {r2_regr_clean:.4f}  |  Theoretical Match ($R^2$): {r2_theory_clean:.4f}")
format_plot(axes[1, 1], title_b, LABEL_X_FORMULA, LABEL_Y_SENSOR)

axes[1, 1].legend(loc='lower right')

plt.show()